# Embeddings

- Any piece of text represented as numerical data is **embeddings** or **vectors**.
- Vectors/embeddings are in the form of arrays, i.e., they are known as **vector arrays**.
- **Dimension**: Size (length) of a vector array.
    - E.g., if dimension size = vector array length = 100, then any string, no matter how long or small, output will be 100 length array.

## OpenAI Embedding Model
- OpenAI provides a model to convert text to embeddings. It provides:
    1. **text-embedding-3-small**: Default dimension size of 1,536.
    2. **text-embedding-3-large**: Default dimension size of 3,072.

## Example:
**Sample Text:**
```js
"To make classic milk tea, bring one cup of water to a boil in a small saucepan and add one teaspoon of black tea leaves along with crushed ginger and cardamom. Let it simmer for two to three minutes so the spices infuse thoroughly into the water. Pour in half a cup of whole milk and add sugar to your taste, then bring the mixture back to a rolling boil over medium heat. Once it bubbles up and turns a rich caramel color, remove it from the heat and strain the hot tea through a fine mesh sieve directly into your cup."
```

**Converting to chunks:**
```js
[
    "To make classic milk tea, bring one cup of water to a boil...",
    "Let it simmer for two to three minutes so the spices infuse...",
    "Pour in half a cup of whole milk and add sugar to your taste, ...",
    "Once it bubbles up and turns a rich caramel color, remove it..."
]
```

**Converting to embeddings:**
```js
[
    "To make classic milk tea..." -> [0.0124, -0.0341, ..., 0.0412],
    "Let it simmer for two..."    -> [-0.0051, 0.0218, ..., -0.0073],
    "Pour in half a cup..."       -> [0.0382, -0.0112, ..., 0.0155],
    "Once it bubbles up..."       -> [-0.0189, 0.0045, ..., 0.0319]
]
```

> **Note:** This is an array of arrays (2D array)

> **Also Note:** The size of every chunk is not equal, still all the embeddings are of same size, i.e., 1536 floats

# Cosine Similarity

Cosine similarity is a measure of how much 2 embeddings are similar.

## Working:

### 1. Upload PDF
- Upload a PDF (e.g, Tea Recipe)
- Break into Chunks
- Convert to embeddings
```javascript
[
    "To make classic milk tea..." -> [0.0124, -0.0341, ..., 0.0412],
    "Let it simmer for two..."    -> [-0.0051, 0.0218, ..., -0.0073],
    "Pour in half a cup..."       -> [0.0382, -0.0112, ..., 0.0155],
    "Once it bubbles up..."       -> [-0.0189, 0.0045, ..., 0.0319]
]
```
---
### 2. Query/Prompt
- Enter a query or prompt (e.g., "How much milk required?")
- Convert it to embedding too.
```javascript
    "How much milk required?" -> [0.0354, -0.0108, ..., 0.0162]
```
- Now, the user query and document both are in the same format: Embedding format
---
### 3. Calculate Cosine Similarity
- Compare the user query embedding with **each** PDF chunk embedding to calculate the **cosine similarity score**.
- Cosine Similarity Scores:
    - **Close to 1:** The meanings of the texts are highly similar.
    - **Close to 0:** The meanings of the texts are neutral or unrelated.
    - **Close to -1:** The meanings of the texts are opposite.

<table>
  <thead>
    <tr>
      <th>Chunk Embedding</th>
      <th>Query Embedding</th>
      <th>Cosine Similarity Score</th>
      <th>Verdict</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>[0.0124, -0.0341, ..., 0.0412]</code></td>
      <td rowspan="4"><code>[0.0354, -0.0108, ..., 0.0162]</code></td>
      <td>0.42</td>
      <td>Near 0 - Neutral</td>
    </tr>
    <tr>
      <td><code>[-0.0051, 0.0218, ..., -0.0073]</code></td>
      <td>0.31</td>
      <td>Near 0 - Neutral</td>
    </tr>
    <tr>
      <td><code>[0.0382, -0.0112, ..., 0.0155]</code></td>
      <td>0.89</td>
      <td>Near 1 - Similar 🤓</td>
    </tr>
    <tr>
      <td><code>[-0.0189, 0.0045, ..., 0.0319]</code></td>
      <td>0.38</td>
      <td>Near 0 - Neutral</td>
    </tr>
  </tbody>
</table>

---
### 4. Choose top_k
- Choose a **top_k** value, means pick the number of similar embeddings to be chosen
- E.g., $k = 2$, then we pick:
```js
[
    0.89 -> [0.0382, -0.0112, ..., 0.0155] -> "Pour in half a cup..." -> Send to LLM
    0.42 -> [0.0124, -0.0341, ..., 0.0412] -> "To make classic milk tea..." -> Send to LLM
]
```
- LLM receives original text chunks, not the raw embedding numbers.
- This top_k value is not chosen randomly, it undergoes an algorithm known as: **Badme dekhenge**

---
### 5. Send to LLM
- Thus, instead of giving 4 chunks of text to the LLM, we provide only the selected 2, reducing cost.
- This was just a simple example of 4 chunks, in reality
    - There can be 1000s of chunks
    - and only a few of them are picked using cosine simlarity
    - which significantly reduces the cost.
---

## Cosine Similarity Formula
$$\cos(\theta) = \frac{A \cdot B}{\Vert{}A\Vert{} \Vert{}B\Vert{}}$$
Where,
- A: Embedding array of user query
- B: Embedding array of a text chunk

# In Python...

In [11]:
from dotenv import load_dotenv
# Not using OpenAIEmbeddings because: I have no money.
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

# Use 'gemini-embedding-001' or 'gemini-embedding-2'
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [12]:
vector1 = embeddings.embed_query("To make classic milk tea, bring one cup of water to a boil in a small saucepan and add one teaspoon of black tea leaves along with crushed ginger and cardamom.")
print(vector1[:6])
print(len(vector1), end="\n")
print(type(vector1))

# vector2 = embeddings.embed_query("Let it simmer for two to three minutes so the spices infuse thoroughly into the water.")
# print(vector2)
# print(len(vector2), end="\n")
# print(type(vector2))

# vector3 = embeddings.embed_query("Pour in half a cup of whole milk and add sugar to your taste, then bring the mixture back to a rolling boil over medium heat.")
# print(vector3)
# print(len(vector3), end="\n")
# print(type(vector3))

# vector4 = embeddings.embed_query("Once it bubbles up and turns a rich caramel color, remove it from the heat and strain the hot tea through a fine mesh sieve directly into your cup.")
# print(vector4)
# print(len(vector4), end="\n")
# print(type(vector4))

[0.006079756, 0.020744408, 0.0065719, -0.054089535, 0.02392944, 0.0035793618]
3072
<class 'list'>
